In [1]:
"""
diagnose_qsa.py  --  localize the  mu_analytic vs mu_overlap  mismatch.
=======================================================================
Run in a notebook next to qsa_section2_circuit.py:

    %run diagnose_qsa.py
    run_all()

It answers ONE question: *where* do circuit and formula diverge? It does NOT
patch anything. Transposing W in the classical formula is exactly the kind of
change that makes a test go green while burying the real cause -- so we localize
first. Note x_j^T W^T x_i = <x_i|W|x_j>, an i<->j swap, which in a CAUSAL sum
(i<=j) is NOT a symmetry: if that "fixes" it, the circuit is computing something
different from the derivation and we need to know what.

Five independent quantities on the same tiny instance:
  mu_overlap    -- circuit, fast overlap readout        (qsa_section2_circuit)
  mu_projector  -- circuit, literal Step-1..4 projector (qsa_section2_circuit)
  mu_numpy      -- faithful numpy statevector emulator  (THIS file, ground truth)
  mu_analytic   -- (1/N^4)|sum a_ij s_ij^k|^2 with s_ij = x_j^T W x_i
  mu_analyticT  -- same but s_ij = x_j^T W^T x_i        (the disputed convention)

Reading the pattern:
  overlap == projector == numpy == analytic          -> all good
  overlap == projector == numpy != analytic          -> W/V matrix EXTRACTION is wrong
                                                        (qml.matrix convention), not the formula
  overlap == projector != numpy == analytic          -> the CIRCUIT differs from the protocol
                                                        (ctrl/StatePrep semantics)
  overlap != projector                               -> the overlap INDEXING is wrong
  analyticT matches while analytic does not          -> a genuine i<->j / transpose convention
                                                        flip: find WHERE, do not just transpose

DIAGNOSTIC INSTANCE MATTERS (verified numerically):
  * d = 2 is DEGENERATE. A 2x2 real-orthogonal W from QR of gaussians is usually a
    REFLECTION (det = -1), which is SYMMETRIC: W == W^T. The transpose is then
    invisible and mu_analytic == mu_analyticT identically. Diagnose at d >= 4.
  * Diagonal terms i=j are transpose-invariant (the scalar x^T W x equals x^T W^T x),
    so only the i<j off-diagonal terms carry the signal: need T >= 2 as well.
  * |mu| alone cannot distinguish the two conventions in the single-pair test
    (both squares are equal) -- test [B] therefore reads the SIGNED amplitude.
"""
from __future__ import annotations
from math import ceil, log2
import numpy as np

# ============================================================================ #
#  0. KNOWN BUG (fix before anything else)
# ============================================================================ #
# qsa_section2_circuit.real_ortho_block does:
#     for a, b in zip(w, w[1:] + w[:1]): qml.CNOT(wires=[a, b])
# With n = 1 (d = 2): w = [0] -> zip([0], [0]) -> CNOT(wires=[0, 0]) -> PennyLane
# raises "Wires must be unique". So any d=2 test CRASHES. Patched version:

def real_ortho_block_fixed(qml, params, wires):
    """RY layers + CNOT ring; entangler skipped when < 2 wires. Real-orthogonal."""
    L = params.shape[0]; w = list(wires)
    for l in range(L):
        for q, wire in enumerate(w):
            qml.RY(params[l, q], wires=wire)
        if len(w) >= 2:                                   # <-- the fix
            ring = zip(w, w[1:] + w[:1]) if len(w) > 2 else [(w[0], w[1])]
            for a, b in ring:
                qml.CNOT(wires=[a, b])


# ============================================================================ #
#  1. numpy GROUND TRUTH: faithful statevector emulation of the protocol
# ============================================================================ #
def _U_first_col(v, seed=0):
    """Real orthogonal U with U[:,0] = v exactly (v real unit)."""
    d = v.shape[0]; rng = np.random.default_rng(seed)
    M = rng.standard_normal((d, d)); M[:, 0] = v
    Q, _ = np.linalg.qr(M); U = Q.copy(); U[:, 0] = v
    # re-orthonormalize columns 1.. against v
    for c in range(1, d):
        u = U[:, c]
        for c2 in range(c):
            u = u - (U[:, c2] @ u) * U[:, c2]
        nrm = np.linalg.norm(u)
        U[:, c] = u / nrm if nrm > 1e-12 else u
    return U

def mu_numpy_emulator(X, Y, Wmat, Vmat, k):
    """Build the FULL statevector over (C1,C2,A,B_1..B_k) as qudits and run
    Steps 1-4 literally. Independent of PennyLane. Ground truth for the formula."""
    T, d = X.shape
    Ntri = T * (T + 1) // 2
    shape = (T, T) + (d,) * (k + 1)
    psi = np.zeros(shape, dtype=float)
    # Step 1: (1/sqrt Ntri) sum_{i<=j} |j,i>|0...0>
    for j in range(T):
        for i in range(j + 1):
            psi[(j, i) + (0,) * (k + 1)] = 1.0 / np.sqrt(Ntri)
    # Step 2: controlled on C2=i, load x_i^{(k+1)} on A,B_1..B_k
    psi2 = np.zeros(shape, dtype=float)
    for j in range(T):
        for i in range(j + 1):
            amp = psi[(j, i) + (0,) * (k + 1)]
            t = X[i]
            for _ in range(k):
                t = np.multiply.outer(t, X[i])
            psi2[(j, i)] = amp * t
    psi = psi2
    # Step 3a: V on A (axis 2), W on each B (axes 3..2+k)
    psi = np.tensordot(Vmat, psi, axes=([1], [2])); psi = np.moveaxis(psi, 0, 2)
    for c in range(k):
        ax = 3 + c
        psi = np.tensordot(Wmat, psi, axes=([1], [ax])); psi = np.moveaxis(psi, 0, ax)
    # Step 3b: controlled on C1=j, U_{y_j}^dag on A and U_{x_j}^dag on each B
    for j in range(T):
        UYd = _U_first_col(Y[j], seed=1000 + j).T
        UXd = _U_first_col(X[j], seed=2000 + j).T
        sl = psi[j]                                        # axes: (C2, A, B_1..B_k)
        sl = np.tensordot(UYd, sl, axes=([1], [1])); sl = np.moveaxis(sl, 0, 1)
        for c in range(k):
            ax = 2 + c
            sl = np.tensordot(UXd, sl, axes=([1], [ax])); sl = np.moveaxis(sl, 0, ax)
        psi[j] = sl
    # Step 4: P^dag then project |0_C>|0_AB>
    A = 0.0
    for j in range(T):
        for i in range(j + 1):
            A += (1.0 / np.sqrt(Ntri)) * psi[(j, i) + (0,) * (k + 1)]
    return float(A ** 2)

def mu_analytic_formula(X, Y, Wmat, Vmat, k, transpose_W=False):
    """(1/N^4)|sum_{i<=j} a_ij s_ij^k|^2 ,  s_ij = x_j^T W x_i  (or W^T if transpose_W)."""
    T = X.shape[0]; Wu = Wmat.T if transpose_W else Wmat
    S = 0.0
    for j in range(T):
        for i in range(j + 1):
            s = float(X[j] @ (Wu @ X[i])); a = float(Y[j] @ (Vmat @ X[i]))
            S += a * s ** k
    Ntri = T * (T + 1) // 2
    return float(S ** 2 / Ntri ** 2)


# ============================================================================ #
#  2. TEST A -- qml.matrix extraction vs hand-built matrix
# ============================================================================ #
def test_matrix_extraction(n=2, L=2, seed=0):
    """Is the W we hand to the formula the SAME W the circuit applies?"""
    import pennylane as qml
    rng = np.random.default_rng(seed)
    Wp = rng.standard_normal((L, n))
    # hand-built: matrix of  [RY layer, CNOT ring] x L,  wire 0 = MSB
    def RY(t): return np.array([[np.cos(t/2), -np.sin(t/2)], [np.sin(t/2), np.cos(t/2)]])
    def kron_list(ms):
        out = np.eye(1)
        for m in ms: out = np.kron(out, m)
        return out
    def cnot_mat(a, b, n):
        dim = 2 ** n; M = np.zeros((dim, dim))
        for x in range(dim):
            bits = [(x >> (n - 1 - w)) & 1 for w in range(n)]
            if bits[a] == 1: bits[b] ^= 1
            y = sum(bit << (n - 1 - w) for w, bit in enumerate(bits))
            M[y, x] = 1.0
        return M
    M = np.eye(2 ** n)
    for l in range(L):
        M = kron_list([RY(Wp[l, q]) for q in range(n)]) @ M
        if n >= 2:
            ring = list(zip(range(n), list(range(1, n)) + [0])) if n > 2 else [(0, 1)]
            for a, b in ring:
                M = cnot_mat(a, b, n) @ M
    Mq = np.real(qml.matrix(real_ortho_block_fixed, wire_order=range(n))(qml, Wp, range(n)))
    same   = np.allclose(M, Mq, atol=1e-8)
    sameT  = np.allclose(M.T, Mq, atol=1e-8)
    print(f"[A] qml.matrix vs hand-built (n={n}): equal={same}  equal_to_TRANSPOSE={sameT}"
          f"  maxdiff={np.abs(M-Mq).max():.2e}")
    if sameT and not same:
        print("    -> EXTRACTION IS TRANSPOSED. This is the bug; fix the extraction, not the formula.")
    return same


# ============================================================================ #
#  3. TEST B -- THE DECISIVE ONE: signed single-pair convention
# ============================================================================ #
def test_single_pair_convention(theta=0.7):
    """n=1, RY(theta) only, x_i=e0, x_j=e1.  Analytically:
         <x_j|W|x_i> = W[1,0] = +sin(theta/2)
         x_j^T W^T x_i = W[0,1] = -sin(theta/2)
    These differ IN SIGN, so reading the signed amplitude localizes the convention.
    (|mu|^2 alone canNOT: both squares are equal.)"""
    import pennylane as qml, jax.numpy as jnp
    W = np.array([[np.cos(theta/2), -np.sin(theta/2)], [np.sin(theta/2), np.cos(theta/2)]])
    e0 = np.array([1.0, 0.0]); e1 = np.array([0.0, 1.0])
    dev = qml.device("default.qubit", wires=1)

    @qml.qnode(dev)
    def amp():                      # StatePrep(e0) -> RY(theta) -> state
        qml.StatePrep(jnp.array(e0), wires=[0])
        qml.RY(theta, wires=0)
        return qml.state()

    st = np.real(np.array(amp()))
    a_circ  = float(e1 @ st)                 # <x_j | W | x_i>  read from the circuit
    a_plain = float(e1 @ W @ e0)             # x_j^T W   x_i  = +sin
    a_tran  = float(e1 @ W.T @ e0)           # x_j^T W^T x_i  = -sin
    print(f"[B] circuit amplitude   = {a_circ:+.6f}")
    print(f"    x_j^T W   x_i       = {a_plain:+.6f}   {'<-- MATCHES' if abs(a_circ-a_plain)<1e-6 else ''}")
    print(f"    x_j^T W^T x_i       = {a_tran:+.6f}   {'<-- MATCHES' if abs(a_circ-a_tran)<1e-6 else ''}")
    if abs(a_circ - a_plain) < 1e-6:
        print("    -> circuit computes <x_j|W|x_i>: the ORIGINAL formula is right; transposing is WRONG.")
    elif abs(a_circ - a_tran) < 1e-6:
        print("    -> circuit computes <x_i|W|x_j>: a genuine i<->j flip lives in the circuit; find it.")
    return a_circ, a_plain, a_tran


# ============================================================================ #
#  4. TEST C -- five-way mu comparison on a tiny instance
# ============================================================================ #
def test_five_way(T=3, d=4, k=2, layers=2, seed=0):
    import qsa_section2_circuit as M
    import pennylane as qml, jax.numpy as jnp
    try:
        M.set_backends()
    except Exception:
        pass
    # patch the n=1 CNOT bug for this session
    M.real_ortho_block = real_ortho_block_fixed

    X, Y, Wp, Vp = M.random_instance(T, d, k, layers, seed)
    n = max(1, ceil(log2(d)))
    Wmat = np.real(qml.matrix(real_ortho_block_fixed, wire_order=range(n))(qml, Wp, range(n)))[:d, :d]
    Vmat = np.real(qml.matrix(real_ortho_block_fixed, wire_order=range(n))(qml, Vp, range(n)))[:d, :d]

    out = {}
    # circuit: overlap readout
    circ_s, ntot = M.make_qsa_state_qnode(T, d, k, layers)
    st = circ_s(jnp.array(X), jnp.array(Y), jnp.array(Wp), jnp.array(Vp))
    ids, N = M.overlap_indices(T, d, k)
    out["mu_overlap"] = M.mu_overlap(st, ids, N)
    # circuit: projector readout
    circ_p, _ = M.make_qsa_projector_qnode(T, d, k, layers)
    out["mu_projector"] = float(circ_p(jnp.array(X), jnp.array(Y), jnp.array(Wp), jnp.array(Vp)))
    # numpy ground truth + both formula conventions
    out["mu_numpy"]     = mu_numpy_emulator(X, Y, Wmat, Vmat, k)
    out["mu_analytic"]  = mu_analytic_formula(X, Y, Wmat, Vmat, k, transpose_W=False)
    out["mu_analyticT"] = mu_analytic_formula(X, Y, Wmat, Vmat, k, transpose_W=True)

    print(f"\n[C] five-way  (T={T} d={d} k={k} n_qubits={ntot})")
    if d == 2:
        print("    NOTE: d=2 is DEGENERATE (2x2 orthogonal W is usually a symmetric")
        print("          reflection, so W == W^T). Cannot diagnose the transpose here.")
    for key, v in out.items():
        print(f"    {key:14s} = {v:.10e}")
    ref = out["mu_overlap"]
    close = {kk: abs(vv - ref) < 1e-8 for kk, vv in out.items()}
    print("\n    agreement with mu_overlap:", {kk: close[kk] for kk in out})
    _verdict(out)
    return out

def _verdict(o, tol=1e-8):
    eq = lambda a, b: abs(o[a] - o[b]) < tol
    print("\n    VERDICT:")
    if not eq("mu_overlap", "mu_projector"):
        print("      overlap != projector  -> the OVERLAP INDEXING is wrong (wire order in")
        print("      overlap_indices). The formula is not implicated.")
        return
    if eq("mu_overlap", "mu_numpy") and eq("mu_overlap", "mu_analytic"):
        print("      everything agrees -- no bug on this instance."); return
    if eq("mu_numpy", "mu_analytic") and not eq("mu_overlap", "mu_numpy"):
        print("      numpy == analytic  BUT  circuit differs -> the PROTOCOL->FORMULA mapping is")
        print("      fine; the CIRCUIT does something else (suspect ctrl(StatePrep)/adjoint")
        print("      semantics). Do NOT transpose the formula.")
        return
    if eq("mu_overlap", "mu_analyticT") and not eq("mu_overlap", "mu_analytic"):
        print("      circuit == analytic(W^T) -> a real i<->j / transpose flip exists.")
        print("      Localize it with test [A] (extraction) and [B] (single-pair). If [A] says")
        print("      the extraction is transposed, fix the EXTRACTION -- transposing the physics")
        print("      formula would be fixing the symptom.")
        return
    print("      mixed pattern -> run [A] and [B]; suspect normalization/dtype (check")
    print("      jax_enable_x64 was set BEFORE device creation) or StatePrep normalization.")


# ============================================================================ #
def run_all():
    print("=" * 74)
    print("QSA diagnostic: localizing mu_analytic vs mu_overlap")
    print("=" * 74)
    try:
        import jax; jax.config.update("jax_enable_x64", True)   # MUST precede device creation
        print("[env] jax_enable_x64 =", jax.config.read("jax_enable_x64"))
    except Exception as e:
        print("[env] jax not available:", e)
    try:
        import pennylane as qml; print("[env] pennylane", qml.__version__)
    except Exception as e:
        print("[env] pennylane not available:", e); return
    print()
    test_single_pair_convention()          # decisive, cheapest
    print()
    for n in (1, 2, 3):
        try:
            test_matrix_extraction(n=n)
        except Exception as e:
            print(f"[A] n={n} raised: {type(e).__name__}: {e}")
    # d>=4 and T>=2 : the only instances that can actually see a transpose flip.
    # k=1 first: for EVEN k, s^k discards the sign, so k=1 is the sharpest probe.
    for (T, d, k) in [(2, 4, 1), (3, 4, 1), (3, 4, 2), (2, 2, 1)]:
        try:
            test_five_way(T=T, d=d, k=k)
        except Exception as e:
            print(f"\n[C] T={T} d={d} k={k} raised: {type(e).__name__}: {e}")

if __name__ == "__main__":
    run_all()

QSA diagnostic: localizing mu_analytic vs mu_overlap
[env] jax_enable_x64 = True
[env] pennylane 0.45.0

[B] circuit amplitude   = +0.342898
    x_j^T W   x_i       = +0.342898   <-- MATCHES
    x_j^T W^T x_i       = -0.342898   
    -> circuit computes <x_j|W|x_i>: the ORIGINAL formula is right; transposing is WRONG.

[A] qml.matrix vs hand-built (n=1): equal=True  equal_to_TRANSPOSE=False  maxdiff=0.00e+00
[A] qml.matrix vs hand-built (n=2): equal=True  equal_to_TRANSPOSE=False  maxdiff=1.11e-16
[A] qml.matrix vs hand-built (n=3): equal=True  equal_to_TRANSPOSE=False  maxdiff=1.11e-16

[C] five-way  (T=2 d=4 k=1 n_qubits=6)
    mu_overlap     = 1.2668487045e-01
    mu_projector   = 1.2668487045e-01
    mu_numpy       = 1.2668487045e-01
    mu_analytic    = 1.2668487045e-01
    mu_analyticT   = 2.7670959112e-01

    agreement with mu_overlap: {'mu_overlap': True, 'mu_projector': True, 'mu_numpy': True, 'mu_analytic': True, 'mu_analyticT': False}

    VERDICT:
      everything agrees -